# OpenAlex affiliation sampling

This notebook generates samples of raw affiliation strings from OpenAlex. Three different sampling strategies are implemented:
1. **Simple random sampling**: all eligible affiliation strings have equal probability of selection.
2. **Weighted random sampling**: affiliation strings that are associated with larger numbers of works have a higher probability of selection.
3. **Institution-based sampling**: affiliation strings are sampled from a large set of institutions selected based on having the largest numbers of works in OpenAlex, ensuring broad institutional coverage.

All samples are drawn from affiliation strings that have been linked to at least one institution in OpenAlex to filter out strings that are unlikely to represent institutional affiliations.

## Initialize GBQ SQL query environment

In [1]:
# Authenticate
from google.colab import auth
auth.authenticate_user()

In [2]:
# Initialize client
from google.cloud import bigquery
from google.cloud.bigquery import magics

project_id = "cwts-leiden"

client = bigquery.Client(project=project_id, location="EU")

magics.context.project = project_id

In [3]:
# Create session
job_config = bigquery.QueryJobConfig(create_session=True)
session_job = client.query("select 1", job_config=job_config)
session_job.result()
session_id = session_job.session_info.session_id

job_config_with_session = bigquery.QueryJobConfig(
    connection_properties=[
        bigquery.query.ConnectionProperty(key="session_id", value=session_id)
    ]
)

magics.context.default_query_job_config = job_config_with_session

## Simple random sampling

In [4]:
n_raw_affiliation_strings = 3000

query = f"""
/*
Simple random sample of raw affiliation strings.

The sampling population consists of raw affiliation strings that have been linked
to at least one institution in OpenAlex. Restricting the population in this way
excludes noisy strings that do not represent institutional affiliations, for
example "None", "Germany", "Berlin", "Independent Researcher",
"View further author information", and "E-mail: [email protected]".

Sampling is performed without weighting, giving all raw affiliation strings an
equal probability of selection.
*/
with raw_affiliation_string as
(
	select a.raw_affiliation_string_id, count(distinct a.work_id) as n_works
	from `cwts-leiden.openalex_2026aug.work_affiliation` as a
	join `cwts-leiden.openalex_2026aug.work_affiliation_institution` as b on a.work_id = b.work_id and a.affiliation_seq = b.affiliation_seq
	group by a.raw_affiliation_string_id
),
weighted_sample as
(
	select raw_affiliation_string_id, n_works, rand() as random_sample_key
	from raw_affiliation_string
	order by random_sample_key
	limit {n_raw_affiliation_strings}
)
select b.raw_affiliation_string
from weighted_sample as a
join `cwts-leiden.openalex_2026aug.raw_affiliation_string` as b on a.raw_affiliation_string_id = b.raw_affiliation_string_id
order by rand()
"""

df_sample1 = client.query(query).to_dataframe()
df_sample1.head(10)

,raw_affiliation_string
0,Research Center for Translational Medicine at ...
1,Department of Clinical Laboratory and Patholog...
2,TETFund Centre of Excellence for Integrated Fa...
3,"Cultural Anthropology, University of Milan-Bic..."
4,"Division of Pediatric Infectious Diseases, The..."
5,"hong,SU Zhong xue,WU Tai huangDepartment of Ge..."
6,"Power Electronics Laboratory, Electrical Engin..."
7,"Department of Public Administration, Payame No..."
8,"Department of General Practice, Utrecht Univer..."
9,"Galera Therapeutics, Inc., 45 Liberty Boulevar..."


## Weighted random sampling

In [5]:
n_raw_affiliation_strings = 3000

query = f"""
/*
Weighted random sample of raw affiliation strings.

The sampling population consists of raw affiliation strings that have been linked
to at least one institution in OpenAlex. Restricting the population in this way
excludes noisy strings that do not represent institutional affiliations.

Sampling probabilities are proportional to the number of distinct works in which
each raw affiliation string occurs. Raw affiliation strings that occur in more
works therefore have a higher probability of selection.

Sampling is performed using the Efraimidis-Spirakis weighted sampling without
replacement algorithm.
*/
with raw_affiliation_string as
(
	select a.raw_affiliation_string_id, count(distinct a.work_id) as n_works
	from `cwts-leiden.openalex_2026aug.work_affiliation` as a
	join `cwts-leiden.openalex_2026aug.work_affiliation_institution` as b on a.work_id = b.work_id and a.affiliation_seq = b.affiliation_seq
	group by a.raw_affiliation_string_id
),
weighted_sample as
(
	select raw_affiliation_string_id, n_works, -ln(rand()) / n_works as weighted_random_sample_key
	from raw_affiliation_string
	order by weighted_random_sample_key
	limit {n_raw_affiliation_strings}
)
select b.raw_affiliation_string
from weighted_sample as a
join `cwts-leiden.openalex_2026aug.raw_affiliation_string` as b on a.raw_affiliation_string_id = b.raw_affiliation_string_id
order by rand()
"""

df_sample2 = client.query(query).to_dataframe()
df_sample2.head(10)

,raw_affiliation_string
0,"Department of Radiation Oncology, Klinikum rec..."
1,German Institute for Economic Research (DIW Be...
2,"School of Mechanical Engineering, Xi’an Univer..."
3,Electrical and Computer Engineering Department...
4,"Department of Biomedical Engineering, Case Wes..."
5,"Analytical Chemistry Section, Division of Canc..."
6,"Johns Hopkins University, Department of Bioche..."
7,"APDM Wearable Technologies, a Clario Company,P..."
8,"Department of Soil and Water Conservation, Nat..."
9,Brigham Young University


## Institution-based random sampling

In [6]:
n_institutions = 3000
n_raw_affiliation_strings_per_institution = 3

query = f"""
/*
Institution-based random sample of raw affiliation strings.

First, the institutions with the largest number of works are selected.

For each selected institution, all associated raw affiliation strings are
identified, and a simple random sample of three raw affiliation strings is
drawn.

This approach ensures broad coverage of institutions while limiting the number
of raw affiliation strings selected for each institution.

The final sample consists of three raw affiliation strings per institution.
*/
with work_affiliation_institution as
(
	select b.work_id, b.affiliation_seq, b.institution_seq, b.institution_id, a.raw_affiliation_string_id
	from `cwts-leiden.openalex_2026aug.work_affiliation` as a
	join `cwts-leiden.openalex_2026aug.work_affiliation_institution` as b on a.work_id = b.work_id and a.affiliation_seq = b.affiliation_seq
),
institution as
(
	select institution_id, count(distinct work_id) as n_works
	from work_affiliation_institution
	group by institution_id
	order by n_works desc
	limit {n_institutions}
),
institution_raw_affiliation_string as
(
	select a.institution_id,
		a.raw_affiliation_string_id,
		count(a.work_id) as n_works,
	from work_affiliation_institution as a
	join institution as b on a.institution_id = b.institution_id
	group by a.institution_id, a.raw_affiliation_string_id
),
institution_raw_affiliation_string2 as
(
	select institution_id,
		raw_affiliation_string_id,
		n_works,
		row_number() over (partition by institution_id order by rand()) as random_sample_rank,
		row_number() over (partition by institution_id order by -ln(rand()) / n_works) as weighted_random_sample_rank
	from institution_raw_affiliation_string
)
select b.raw_affiliation_string
from institution_raw_affiliation_string2 as a
join `cwts-leiden.openalex_2026aug.raw_affiliation_string` as b on a.raw_affiliation_string_id = b.raw_affiliation_string_id
where random_sample_rank <= {n_raw_affiliation_strings_per_institution}
order by rand()
"""

df_sample3 = client.query(query).to_dataframe()
df_sample3.head(10)

,raw_affiliation_string
0,"†Department of special education, University o..."
1,Sebelas Maret UniversityUPGRIS
2,School ofEngineering ， The Univ ． of Tokyo Hon...
3,"Département Informatique, Université de Pau et..."
4,"Université de Caen Normandie, Institut Nationa..."
5,Hefei National Laboratory of Physical Sciences...
6,"School of Medicine, Queen’s University, Kingst..."
7,"Institute of Materials Structure Science, The ..."
8,"Assistant Professor, Dept. of Surgery, Autonom..."
9,"Department of Engineering Physics, Faculty of ..."


## Export results

In [7]:
import csv

df_sample1.to_csv('openalex_affiliation_sample1.txt', index=False, encoding="utf-8-sig", sep='\t', quoting=csv.QUOTE_NONE)
df_sample2.to_csv('openalex_affiliation_sample2.txt', index=False, encoding="utf-8-sig", sep='\t', quoting=csv.QUOTE_NONE)
df_sample3.to_csv('openalex_affiliation_sample3.txt', index=False, encoding="utf-8-sig", sep='\t', quoting=csv.QUOTE_NONE)